# 22 — Separat submission: 90 % LP-blend + 10 % v5-blend

**Henrik Remmen (541747), gruppe 119 — [119] SINTEF Databreach**

Dette er **ikke** 90 % XGBoost + 10 % LightGBM. Begge inputfilene er allerede
50/50-blender av XGBoost og LightGBM:

- **90 %** `submissions/hydro_lp_v6.csv` — notebook 19, med LP-features.
- **10 %** `submissions/multiscale_boosting_v5.csv` — notebook 18, uten LP-features.

For hver generator/time: `p = 0.90 * p_LP + 0.10 * p_v5`.
Ingen trening, ingen vektsøk, ingen nettverkskall og ingen automatisk Kaggle-opplasting.

**Originalene er read-only.** Rader matches på `Run No`, kolonner på navn, og
output følger `sample_submission.csv`. Verdiene avrundes ikke til 0/1.

Notebook 21 viste en liten AUC-nedgang i 2021 og forbedring i 2022 for denne vekten.
Dette er en **eksperimentell reservekandidat**, ikke en dokumentert bedre erstatning for LP.
Vekten er låst før ny public-score.

Denne notebooken trenger eksisterende predictions og er derfor **ikke tilstrekkelig som
professorens kortnotebook**. En eventuell endelig leveranse må gjenskape begge grunnmodellene
fra de utdelte rådataene.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import uuid
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/"sample_submission.csv").is_file() and (p/"submissions").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Kjør notebooken i prosjektet med submissions/ og sample_submission.csv.")

LP_PATH = ROOT/"submissions/hydro_lp_v6.csv"
V5_PATH = ROOT/"submissions/multiscale_boosting_v5.csv"
SAMPLE_PATH = ROOT/"sample_submission.csv"
LP_WEIGHT, V5_WEIGHT = 0.90, 0.10  # Fast hypotese fra notebook 21.
for path in (LP_PATH, V5_PATH, SAMPLE_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Mangler {path}; notebook 18 og 19 må være ferdige.")
print("90 %:", LP_PATH.name)
print("10 %:", V5_PATH.name)


90 %: hydro_lp_v6.csv
10 %: multiscale_boosting_v5.csv


In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_frame(path):
    # Kontroller header FØR pandas kan omdøpe dupliserte kolonnenavn.
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        header = next(csv.reader(stream))
    if len(header) != len(set(header)):
        raise ValueError(f"Dupliserte kolonner i {path.name}")
    return pd.read_csv(path, dtype={"Run No": "string"}, encoding="utf-8-sig")

def align_predictions(frame, sample, name):
    if "Run No" not in frame or "Run No" not in sample:
        raise ValueError("Run No mangler.")
    if frame.columns.has_duplicates or sample.columns.has_duplicates:
        raise ValueError("Dupliserte kolonner.")
    for label, f in ((name,frame),("sample",sample)):
        if f["Run No"].isna().any() or f["Run No"].duplicated().any():
            raise ValueError(f"{label}: tom eller duplisert Run No.")
    if frame.shape != sample.shape or set(frame.columns) != set(sample.columns):
        raise ValueError(f"{name}: feil antall rader/kolonner eller kolonnenavn.")
    if set(frame["Run No"]) != set(sample["Run No"]):
        raise ValueError(f"{name}: Run No stemmer ikke med sample_submission.")
    cols = [c for c in sample.columns if c != "Run No"]
    aligned = frame.set_index("Run No").loc[sample["Run No"], cols]
    values = aligned.to_numpy(dtype=np.float64)
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise ValueError(f"{name}: ugyldige sannsynligheter.")
    return values

# Alle eksisterende submission-CSV-er beskyttes, ikke bare de to inputfilene.
protected = sorted(set((ROOT/"submissions").glob("*.csv")) | {SAMPLE_PATH})
ORIGINAL_HASHES = {str(p.relative_to(ROOT)):sha256(p) for p in protected}
sample, lp, v5 = (read_frame(p) for p in (SAMPLE_PATH, LP_PATH, V5_PATH))
lp_values = align_predictions(lp, sample, "LP")
v5_values = align_predictions(v5, sample, "v5")
prediction_columns = [c for c in sample.columns if c != "Run No"]
if len(sample) != 725 or len(prediction_columns) != 14*168:
    raise ValueError("Forventet dette datasettets 725 case og 14 × 168 prediksjoner.")

blended_values = LP_WEIGHT*lp_values + V5_WEIGHT*v5_values
blend = pd.DataFrame(blended_values, columns=prediction_columns)
blend.insert(0, "Run No", sample["Run No"].to_numpy())
blend = blend.loc[:, sample.columns]
align_predictions(blend, sample, "ny blend")
display(blend.iloc[:3,:6])
print("Kontrollert form:", blend.shape)
print("Gjennomsnittlig absolutt endring fra LP:", float(np.abs(blended_values-lp_values).mean()))
print("Ingen fil er skrevet ennå.")


,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4
0,2923,0.059629,0.028355,0.008287,0.004468,0.005280
1,2924,0.772240,0.770284,0.779369,0.794858,0.809534
2,2925,0.999677,0.999705,0.999748,0.999772,0.999707


Kontrollert form: (725, 2353)
Gjennomsnittlig absolutt endring fra LP: 0.003987692526511904
Ingen fil er skrevet ennå.


## Skriv én NY CSV og en separat manifestfil

Før skriving kontrolleres originalfilene på nytt. Hvis standardnavnet allerede finnes,
får den nye CSV-en et unikt suffiks. Eksklusiv filoppretting (`x`) hindrer overskriving.

Etterpå leses den nye filen tilbake, og identifikatorer, format og verdier kontrolleres.
Manifestet er dokumentasjon, **ikke** en fil som skal lastes opp til Kaggle.


In [3]:
def verify_originals():
    changed = [name for name, fingerprint in ORIGINAL_HASHES.items()
               if not (ROOT/name).is_file() or sha256(ROOT/name) != fingerprint]
    if changed:
        raise RuntimeError(f"Originalfiler har endret seg siden innlasting: {changed}")

verify_originals()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:8]
OUTPUT = ROOT/"submissions/hydro_lp_v6_v5_blend_90_10.csv"
if OUTPUT.exists():
    OUTPUT = OUTPUT.with_name(f"{OUTPUT.stem}_{RUN_ID}.csv")
if OUTPUT.resolve() in {p.resolve() for p in protected}:
    raise ValueError("Output kan ikke være en originalfil.")

# 'x' feiler hvis filen finnes: aldri overskriv.
with OUTPUT.open("x", encoding="utf-8", newline="") as stream:
    blend.to_csv(stream, index=False, float_format="%.17g")

saved = read_frame(OUTPUT)
saved_values = align_predictions(saved, sample, "lagret blend")
if not saved["Run No"].equals(sample["Run No"]) or list(saved.columns) != list(sample.columns):
    raise AssertionError("Lagret rekkefølge avviker fra sample_submission.")
np.testing.assert_allclose(saved_values, blended_values, rtol=0, atol=1e-14)
verify_originals()

RECORD_DIR = ROOT/"artifacts/submission_blend_90_10"/RUN_ID
RECORD_DIR.mkdir(parents=True, exist_ok=False)
manifest = dict(
    run_id=RUN_ID, method="probability_average", weights={"LP":LP_WEIGHT,"v5":V5_WEIGHT},
    inputs={str(p.relative_to(ROOT)):ORIGINAL_HASHES[str(p.relative_to(ROOT))]
            for p in (LP_PATH,V5_PATH,SAMPLE_PATH)},
    output=str(OUTPUT.relative_to(ROOT)), output_sha256=sha256(OUTPUT),
    rows=len(saved), prediction_columns=len(prediction_columns),
    original_csv_unchanged=True, protected_original_csv_sha256=ORIGINAL_HASHES,
    packages={"numpy":np.__version__,"pandas":pd.__version__},
    note="Experimental backup; frozen 90/10 from notebook 21, not tuned to new public score.")
with (RECORD_DIR/"blend_manifest.json").open("x", encoding="utf-8") as stream:
    json.dump(manifest, stream, indent=2)

print("FERDIG — last opp denne CSV-en til Kaggle:")
print(OUTPUT)
print(f"{len(ORIGINAL_HASHES)} originale CSV-er er kontrollert uendret.")
print("Manifest (ikke Kaggle-submission):", RECORD_DIR/"blend_manifest.json")
print("Behold hydro_lp_v6.csv uendret som hovedkandidat. Lagre notebooken med Cmd+S.")


FERDIG — last opp denne CSV-en til Kaggle:
/Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/submissions/hydro_lp_v6_v5_blend_90_10.csv
32 originale CSV-er er kontrollert uendret.
Manifest (ikke Kaggle-submission): /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/submission_blend_90_10/20260928T235125Z_3dd49714/blend_manifest.json
Behold hydro_lp_v6.csv uendret som hovedkandidat. Lagre notebooken med Cmd+S.
